# Motor insurance pricing with GLMs

This notebook uses **real freMTPL2 data** exported using `Rscript r/01_export_pricing_data.R`. Run from the project root. The model is an educational pricing indication, not an insurer-approved rate.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from python.pricing import prepare_data, fit_models, predict_prices

## 1. Load and inspect the data
`Exposure` is measured in policy-years; `ClaimNb` is the reported claim count.

In [ ]:
freq = pd.read_csv('data/freMTPL2freq.csv')
sev = pd.read_csv('data/freMTPL2sev.csv')
df = prepare_data(freq, sev)
print('Policies:', len(df))
print('Claim count:', df.ClaimNb.sum())
print('Observed claims cost:', round(df.ClaimAmount.sum(), 2))
display(df.head())

## 2. Portfolio statistics

In [ ]:
portfolio_frequency = df.ClaimNb.sum() / df.Exposure.sum()
portfolio_severity = df.ClaimAmount.sum() / df.ClaimNb.sum()
print(f'Observed claims per policy-year: {portfolio_frequency:.4f}')
print(f'Observed cost per reported claim: {portfolio_severity:,.2f}')
print(f'Observed pure premium per policy-year: {portfolio_frequency * portfolio_severity:,.2f}')

## 3. Fit GLMs
Poisson frequency includes a log-exposure offset. Gamma severity uses a log link and claim-count weights.

In [ ]:
frequency_model, severity_model = fit_models(df)
print(frequency_model.summary())
print(severity_model.summary())

## 4. Estimate pure premiums
Pure premium = annualised expected frequency × expected claim severity. A 25% expense ratio is an illustrative assumption.

In [ ]:
prices = predict_prices(df, frequency_model, severity_model)
display(prices[['IDpol','expected_frequency','expected_severity','pure_premium','illustrative_premium']].head())

## 5. Compare observed and modelled costs by vehicle brand
Portfolio summaries are exposure-weighted; investigate groups with poor calibration.

In [ ]:
by_brand = prices.groupby('VehBrand').apply(lambda x: pd.Series({
    'observed': x.ClaimAmount.sum() / x.Exposure.sum(),
    'modelled': (x.pure_premium * x.Exposure).sum() / x.Exposure.sum()
}), include_groups=False)
by_brand.plot(kind='bar', figsize=(10,4), title='Observed vs modelled pure premium by vehicle brand')
plt.ylabel('Claim cost per exposure-year')
plt.tight_layout()
plt.show()

## 6. Discussion
- Which rating factors appear material?
- How does the Poisson variance assumption compare with observed count dispersion?
- Is the Gamma severity assumption reasonable for large claims?
- What happens to the premium indication if expense assumptions change?

**Limitations:** In-sample comparison is not out-of-sample validation. This simplified portfolio model does not include inflation, reinsurance, credibility or regulatory constraints.